In [11]:
import pandas as pd
import os

# Create KPI outputs folder if not exists
os.makedirs("KPI outputs", exist_ok=True)



In [39]:
customer = pd.read_csv('C:Cleaned data\\customer.csv')
inventory = pd.read_csv('C:Cleaned data\\inventory.csv')
orders = pd.read_csv('C:Cleaned data\\orders.csv')
products = pd.read_csv('C:Cleaned data\\products.csv')
order_items = pd.read_csv('C:Cleaned data\\order_items.csv')
payment = pd.read_csv('C:Cleaned data\\payment.csv')
returns = pd.read_csv('C:Cleaned data\\returns.csv')
supplier = pd.read_csv('C:Cleaned data\\supplier.csv')

In [13]:
# Convert date columns to datetime
orders['order_date']   = pd.to_datetime(orders['order_date'], errors='coerce')
payment['payment_time'] = pd.to_datetime(payment['payment_time'], errors='coerce')
returns['return_date']  = pd.to_datetime(returns['return_date'], errors='coerce')

# Quick check
print("Orders dtype:", orders['order_date'].dtype)
print(orders['order_date'].head())


Orders dtype: datetime64[us]
0   2025-05-22
1   2023-10-16
2   2023-10-08
3   2023-10-02
4   2024-06-15
Name: order_date, dtype: datetime64[us]


In [14]:
monthly_trend = orders.groupby(orders['order_date'].dt.to_period('M')).agg(
    total_orders=('order_id','count'),
    gross_revenue_inr=('final_amount','sum'),
    total_gst_collected=('gst_amount','sum'),
    total_discounts=('discount_amount','sum'),
    avg_order_value=('final_amount','mean'),
    cancelled_orders=('status', lambda x: (x=='Cancelled').sum())
).reset_index()

monthly_trend.to_csv("KPI outputs/monthly_revenue_trend.csv", index=False)


In [15]:
category_perf = order_items.groupby('category').agg(
    orders=('order_id','nunique'),
    units_sold=('quantity','sum'),
    revenue_inr=('total_price','sum'),
    avg_selling_price=('unit_price','mean'),
    gst_collected=('gst_amount','sum')
).reset_index()

category_perf['revenue_share_pct'] = (category_perf['revenue_inr'] / category_perf['revenue_inr'].sum())*100
category_perf.sort_values('revenue_inr', ascending=False).to_csv("KPI outputs/category_sales_perf.csv", index=False)


In [16]:
segment_analysis = customer.groupby('segment').agg(
    customer_count=('customer_id','count'),
    avg_orders=('total_orders','mean'),
    avg_lifetime_value_inr=('total_spent','mean'),
    segment_revenue_inr=('total_spent','sum'),
    avg_age=('age','mean')
).reset_index()

segment_analysis.to_csv("KPI outputs/customer_segment_analysis.csv", index=False)


In [18]:
payment_dist = payment.groupby('payment_method').agg(
    transactions=('payment_method','count'),
    total_amount_inr=('amount','sum'),
    avg_transaction_value=('amount','mean'),
    failed_count=('status', lambda x: (x=='Failed').sum())
).reset_index()

payment_dist['failure_rate_pct'] = (payment_dist['failed_count']/payment_dist['transactions'])*100
payment_dist.to_csv("KPI outputs/payment_method_distribution.csv", index=False)


In [17]:
returns_cat = order_items.merge(returns, on='order_id', how='left')
return_rate = returns_cat.groupby('category').agg(
    returns=('return_id','nunique'),
    orders=('order_id','nunique'),
    return_value_inr=('return_amount','sum')
).reset_index()

return_rate['return_rate_pct'] = (return_rate['returns']/return_rate['orders'])*100
return_rate.to_csv("KPI outputs/return_rate_by_category.csv", index=False)


In [34]:
top_products = order_items.merge(products, on='product_id', suffixes=('_oi','_p')).merge(orders, on='order_id')
print(top_products.columns)

Index(['item_id', 'order_id', 'product_id', 'product_name_oi', 'category_oi',
       'quantity', 'unit_price', 'gst_rate_oi', 'gst_amount_x',
       'discount_amount_x', 'total_price', 'product_name_p', 'category_p',
       'subcategory', 'brand', 'sku', 'mrp', 'selling_price', 'cost_price',
       'gst_rate_p', 'hsn_code', 'weight_grams', 'supplier_id', 'rating',
       'review_count', 'is_active', 'launch_date', 'customer_id', 'order_date',
       'order_time', 'status', 'city', 'state', 'pincode', 'total_amount',
       'gst_amount_y', 'shipping_charge', 'discount_amount_y', 'final_amount',
       'payment_method', 'shipping_partner', 'tracking_id', 'delivered_date',
       'is_cod', 'channel', 'day_of_week'],
      dtype='str')


In [36]:
# Filter only delivered orders
top_products = top_products[top_products['status'] == 'Delivered']

# Group by correct product columns
top10 = top_products.groupby(['product_id','product_name_p','category_p','brand']).agg(
    units_sold=('quantity','sum'),
    revenue_inr=('total_price','sum'),
    product_rating=('rating','mean')
).reset_index().sort_values('revenue_inr', ascending=False).head(10)

# Save output
import os
os.makedirs("KPI outputs", exist_ok=True)
top10.to_csv("KPI outputs/top10_products.csv", index=False)

print(top10)


   product_id                        product_name_p   category_p      brand  \
85    PRD0086                   Godrej Camera Elite  Electronics     Godrej   
66    PRD0067            Ambrane Power Bank Classic  Electronics    Ambrane   
61    PRD0062            Godrej Power Bank Standard  Electronics     Godrej   
45    PRD0046           Pidilite Headphones Premium  Electronics   Pidilite   
54    PRD0055   Reliance Bluetooth Speaker Standard  Electronics   Reliance   
53    PRD0054  Zebronics Bluetooth Speaker Standard  Electronics  Zebronics   
38    PRD0039                     Hero Smart TV Max  Electronics       Hero   
17    PRD0018                  Infosys Laptop Ultra  Electronics    Infosys   
98    PRD0099                  Bajaj Router Classic  Electronics      Bajaj   
76    PRD0077                  Mivi Smartwatch Plus  Electronics       Mivi   

    units_sold  revenue_inr  product_rating  
85         181  27804029.78             3.8  
66         160  25397714.76           

In [37]:
state_rev = orders[orders['status']=='Delivered'].groupby('state').agg(
    customers=('customer_id','nunique'),
    orders=('order_id','count'),
    revenue_inr=('final_amount','sum'),
    aov_inr=('final_amount','mean')
).reset_index()

state_rev.to_csv("KPI outputs/statewise_revenue.csv", index=False)


In [42]:
# Add a flag column for below reorder
inventory['below_reorder_flag'] = (inventory['quantity_available'] < inventory['reorder_level']).astype(int)

# Now groupby safely
inventory_health = inventory.groupby(['warehouse_location','status']).agg(
    product_count=('product_id','count'),
    total_units=('quantity_available','sum'),
    inventory_value_inr=('total_inventory_value','sum'),
    below_reorder=('below_reorder_flag','sum')
).reset_index()

# Save output
import os
os.makedirs("KPI outputs", exist_ok=True)
inventory_health.to_csv("KPI outputs/inventory_health.csv", index=False)

print(inventory_health.head())


  warehouse_location        status  product_count  total_units  \
0      Bengaluru-WH3      In Stock            154        38490   
1      Bengaluru-WH3     Low Stock             16          281   
2      Bengaluru-WH3  Out of Stock              1            0   
3        Chennai-WH5      In Stock            178        47229   
4        Chennai-WH5     Low Stock             11          171   

   inventory_value_inr  below_reorder  
0            284195630              0  
1              1717611             16  
2                    0              1  
3            406688434              0  
4               227038             11  


In [44]:
supplier_perf = products.merge(supplier, on='supplier_id', suffixes=('_p','_s'))
print(supplier_perf.columns)

Index(['product_id', 'product_name', 'category_p', 'subcategory', 'brand',
       'sku', 'mrp', 'selling_price', 'cost_price', 'gst_rate', 'hsn_code',
       'weight_grams', 'supplier_id', 'rating_p', 'review_count',
       'is_active_p', 'launch_date', 'supplier_name', 'contact_person',
       'email', 'phone', 'city', 'state', 'pincode', 'category_s', 'gstin',
       'payment_terms_days', 'rating_s', 'created_date', 'is_active_s'],
      dtype='str')


In [47]:
# Group by supplier details
supplier_perf_summary = supplier_perf.groupby(
    ['supplier_id','supplier_name','category_s','rating_s','payment_terms_days']
).agg(
    products_supplied=('product_id','count'),
    avg_product_rating=('rating_p','mean')
).reset_index()

# Save output
import os
os.makedirs("KPI outputs", exist_ok=True)
supplier_perf_summary.to_csv("KPI outputs/supplier_performance.csv", index=False)

print(supplier_perf_summary.head())



  supplier_id           supplier_name        category_s  rating_s  \
0     SUP0001       Iyer Elec Pvt Ltd       Electronics       4.1   
1     SUP0002     Sharma Offi Pvt Ltd   Office Supplies       4.5   
2     SUP0003  Choudhury Beau Pvt Ltd   Beauty & Health       4.3   
3     SUP0005      Patil Spor Pvt Ltd  Sports & Fitness       5.0   
4     SUP0006  Choudhury Fash Pvt Ltd           Fashion       4.5   

   payment_terms_days  products_supplied  avg_product_rating  
0                  15                  5            3.860000  
1                  15                  6            4.383333  
2                  60                  8            3.887500  
3                  30                  4            4.175000  
4                  45                  6            3.733333  


In [49]:
customer['cohort_month'] = pd.to_datetime(customer['registration_date'], errors='coerce').dt.to_period('M')
cohort = customer.groupby(['cohort_month','segment']).agg(
    new_customers=('customer_id','count'),
    orders_placed=('total_orders','sum'),
    revenue_inr=('total_spent','sum')
).reset_index()

cohort.to_csv("KPI outputs/cohort_analysis.csv", index=False)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_22204\2945936099.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  customer['cohort_month'] = pd.to_datetime(customer['registration_date'], errors='coerce').dt.to_period('M')


In [50]:
channel_perf = orders.groupby('channel').agg(
    orders=('order_id','count'),
    revenue_inr=('final_amount','sum'),
    avg_order_value=('final_amount','mean'),
    cancel_rate=('status', lambda x: (x=='Cancelled').sum()*100/len(x))
).reset_index()

channel_perf.to_csv("KPI outputs/channel_performance.csv", index=False)


In [52]:
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')


In [53]:
orders['year'] = orders['order_date'].dt.year
orders['month'] = orders['order_date'].dt.month


In [55]:
festive = orders[orders['month'].isin([10,11])].groupby(['year','month']).agg(
    orders=('order_id','count'),
    revenue_inr=('final_amount','sum')
).reset_index()

os.makedirs("KPI outputs", exist_ok=True)
festive.to_csv("KPI outputs/festive_sales.csv", index=False)

print(festive)

   year  month  orders   revenue_inr
0  2023     10    2060  1.324709e+08
1  2023     11    2114  1.372634e+08
2  2024     10    2131  1.335183e+08
3  2024     11    2084  1.262641e+08


In [56]:
#save all files in Cleaned data folder
customer.to_csv("Cleaned data/customer.csv",index=False)
inventory.to_csv("Cleaned data/inventory.csv",index=False)
orders.to_csv("Cleaned data/orders.csv",index=False)
products.to_csv("Cleaned data/products.csv",index=False)
order_items.to_csv("Cleaned data/order_items.csv",index=False)
payment.to_csv("Cleaned data/payment.csv",index=False)
returns.to_csv("Cleaned data/returns.csv",index=False)
supplier.to_csv("Cleaned data/supplier.csv",index=False)